# 01 · Build the CLIP feature cache

Encodes every ID and OOD image **once** with CLIP ViT-B/16 (fp16, exactly like OpenOOD-VLM) and computes
all text-side features. Every later notebook replays test streams from this cache — no images needed.

**Settings:** GPU T4 ×2 / P100 · Internet On. **Inputs:** `oodlab-code`, the ImageNet competition, and
(optional) a dataset with OpenOOD zips if Google Drive refuses downloads (GUIDE.md §5).
**Time:** ≈ 1–2 h (mostly JPEG decoding and downloads). **Run it with "Save Version → Save & Run All"**
so it survives a closed browser.

**Output** (`/kaggle/working/cache/vit-b-16/`, ≈ 0.3 GB):

| file | content |
|---|---|
| `textbank.pt` | 1,000 ID prompts ("The nice …", "a photo of a …"), the 69,554-word WordNet corpus in the official layout, NegLabel's 10,000 mined negatives, 15 noise-image features |
| `images/<set>.npz` | fp16 features, labels (−1 = OOD), image keys |
| `manifests/<set>.csv` | the exact image list that was encoded |

**If the session dies:** add this notebook's own previous output as an input and run again — finished
sets are imported and unfinished ones resume from their last 10k-image shard.

In [ ]:
# ── Set-up 1/2: find the oodlab code (attached as a private Kaggle dataset) ─────────
import os, sys

def find_oodlab(roots, max_depth=4):
    """Bounded breadth-first search for a folder containing oodlab/__init__.py.
    Never walks image folders (the old notebook hung on a recursive glob over ImageNet)."""
    skip = {"ILSVRC", "train", "val", "test", "images", "__pycache__", "cache", "results", "logs"}
    for root in [r for r in roots if r and os.path.isdir(r)]:
        frontier = [(root, 0)]
        while frontier:
            d, depth = frontier.pop(0)
            if os.path.isfile(os.path.join(d, "oodlab", "__init__.py")):
                return d
            if depth < max_depth:
                try:
                    subs = sorted(e.path for e in os.scandir(d)
                                  if e.is_dir(follow_symlinks=False) and e.name not in skip)
                except OSError:
                    continue
                frontier += [(s, depth + 1) for s in subs]
    return None

ROOTS = [os.environ.get("OODLAB_CODE"), "/kaggle/input", "/kaggle/working", os.getcwd()]
CODE_ROOT = find_oodlab(ROOTS)
if CODE_ROOT is None:   # the dataset was uploaded as a zip that Kaggle did not unpack -> unpack it ourselves
    import zipfile
    for root in [r for r in ROOTS[:2] if r and os.path.isdir(r)]:
        for dp, dn, fns in os.walk(root):
            dn[:] = [d for d in dn if d not in {"ILSVRC", "train", "val", "test", "images"}] if dp.count(os.sep) - root.count(os.sep) < 3 else []
            for fn in fns:
                if fn.startswith("oodlab") and fn.endswith(".zip"):
                    zipfile.ZipFile(os.path.join(dp, fn)).extractall("/kaggle/working/_oodlab_code")
    CODE_ROOT = find_oodlab(["/kaggle/working/_oodlab_code"])
if CODE_ROOT is None:
    raise FileNotFoundError("oodlab code not found: attach your private dataset with the code "
                            "(Add Input -> Your Datasets -> oodlab-code). See GUIDE.md, section 2.")
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)
import oodlab
print("oodlab", oodlab.__version__, "loaded from", CODE_ROOT)

In [ ]:
# ── Set-up 2/2: packages, folders, logging ─────────────────────────────────────────
from oodlab.session import start, finish, ensure_packages
ensure_packages(clip=True, gdown=True)      # CLIP ships inside oodlab; this only adds small pip packages if missing
ctx = start("01_build_feature_cache", need_gpu=True)
log = ctx.log                                   # log.info(...) -> screen + /kaggle/working/logs/01_build_feature_cache.log
import numpy as np, pandas as pd, torch, time, json
from oodlab.report import show, save_tables, write_report

### Settings

In [ ]:
BACKBONE = "ViT-B/16"
SETS = ["imagenet_val_all", "imagenet_test", "imagenet_val",      # ID (Kaggle ImageNet val)
        "ninco", "textures_all", "textures", "inaturalist", "sun", "places",
        "openimage_o", "openimage_o_val", "ssb_hard"]               # OOD (OpenOOD v1.5 downloads)
BATCH_SIZE = 256
NUM_WORKERS = min(4, os.cpu_count() or 2)   # Kaggle GPU machines have 4 vCPUs
SHARD_SIZE = 10_000
TOTAL_NEG = 10_000                           # NegLabel's mined negatives (official ood_number)
LIVE_CHECK = True                            # official TANL on live images vs our TANL on the cache
LIVE_CHECK_N_ID = 5_000
CORPUS_LIMIT = None
# If Google Drive refuses a file, copy it to your own Drive ("Make a copy"), share it with
# "anyone with the link", and put the copy's id here, e.g. {"ssb_hard": "1AbC..."}  (GUIDE.md §5)
DRIVE_ID_OVERRIDES = {}
from oodlab.data.openood import DRIVE_IDS
DRIVE_IDS.update(DRIVE_ID_OVERRIDES)
if ctx.smoke:
    BATCH_SIZE, NUM_WORKERS, SHARD_SIZE, TOTAL_NEG, LIVE_CHECK_N_ID, CORPUS_LIMIT = 32, 0, 150, 500, 100, 2000

### 1 · Resume: import anything a previous run already finished

In [ ]:
from oodlab.cache import FeatureCache, find_existing_caches, import_from, encode_manifest
cache = FeatureCache(ctx.dirs.cache, BACKBONE).make()
for root in find_existing_caches([ctx.dirs.inputs]):
    import_from(root, cache, logger=log)
log.info(f"cache at {cache.dir}; already done: {cache.available()} textbank={cache.has_textbank()}")

### 2 · CLIP

In [ ]:
from oodlab.clipwrap import load_clip
model = load_clip(BACKBONE, device=ctx.device, input_roots=ctx.input_roots, random_init=ctx.smoke)
log.info(f"CLIP {BACKBONE}: dtype {model.dtype}, logit scale {model.logit_scale.exp().item():.6f}")

### 3 · Text bank (≈ 1 min on GPU)

Word lists are NegLabel's WordNet files shipped inside the package (`oodlab/resources/neglabel_txtfiles`,
Apache-2.0). De-duplication follows the official code and must give **11,453 adjectives + 58,101 nouns
= 69,554 words**; NegMining (95th-percentile cosine to the ID names, ViT-B/16) keeps **1,646 + 8,354**.

In [ ]:
from oodlab.textbank import build_textbank, OFFICIAL_N_ADJ, OFFICIAL_N_NOUN
if not cache.has_textbank():
    bank = build_textbank(model, BACKBONE, oodlab.imagenet_classnames(), oodlab.CORPUS_DIR,
                          total_neg=TOTAL_NEG, corpus_limit=CORPUS_LIMIT, logger=log)
    cache.save_textbank(bank)
bank = cache.load_textbank()
print(json.dumps(bank.summary(), indent=1))
st = bank.meta["stats"]
if not ctx.smoke:
    assert (st["n_adj"], st["n_noun"]) == (OFFICIAL_N_ADJ, OFFICIAL_N_NOUN), st
    assert bank.n_sel_adj == 1646 and bank.n_selected == 10000, (bank.n_sel_adj, bank.n_selected)
print("most OOD-like selected words:", bank.words[:8], bank.words[bank.n_sel_adj:bank.n_sel_adj + 8])
print("closest-to-ID unselected words:", bank.words[-8:])

### 4 · Image lists and images, one set at a time

For each set: build the manifest (downloading and extracting the OpenOOD zip if needed), check files
exist, encode, save, and delete the extracted images to keep disk use low. A failure (e.g. a Drive
quota error) is logged and the loop moves on; re-run the notebook later to fill the gaps.

In [ ]:
from oodlab.data import build_manifest, SPECS
from oodlab.data.openood import obtain_imglist_dir, cleanup
try:
    imglist_dir = obtain_imglist_dir(ctx.dirs.scratch, ctx.input_roots, logger=log)
    log.info(f"OpenOOD image lists: {imglist_dir}: {sorted(os.listdir(imglist_dir))}")
except Exception as e:   # sets that need a list will fail individually below; the others still run
    log.exception("OpenOOD image lists unavailable (GUIDE.md §5)")
    imglist_dir = None

def cleanup_images(key):
    users = [n for n in SETS if SPECS[n].drive_key == key]
    if key and all(cache.has(n) for n in users) and not (LIVE_CHECK and key == "ninco"):
        cleanup(os.path.join(ctx.dirs.scratch, "extracted", key))
        cleanup(os.path.join(ctx.dirs.scratch, "zips", f"{key}.zip"))

status, imagenet_memo = [], {}
for name in SETS:
    t0 = time.time()
    if cache.has(name):
        status.append({"set": name, "status": "cached", "images": len(cache.load_set(name)), "min": 0.0})
        continue
    try:
        man = build_manifest(name, ctx.input_roots, ctx.dirs.scratch, imglist_dir, logger=log,
                             strict_counts=not ctx.smoke, _imagenet_cache=imagenet_memo)
        missing = man.missing_files(sample=500)
        if missing:
            raise FileNotFoundError(f"{len(missing)} of 500 sampled files missing, e.g. {missing[:2]}")
        fs = encode_manifest(model, man, cache, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                             shard_size=SHARD_SIZE, logger=log)
        status.append({"set": name, "status": "encoded", "images": len(fs), "expected": SPECS[name].expected,
                       "min": round((time.time() - t0) / 60, 1)})
    except Exception as e:
        log.exception(f"[{name}] FAILED")
        status.append({"set": name, "status": f"FAILED: {type(e).__name__}: {str(e)[:150]}", "min": round((time.time() - t0) / 60, 1)})
    finally:
        cleanup_images(SPECS[name].drive_key)
    log.info(f"[{name}] {status[-1]['status']} ({ctx.elapsed()} since start)")
show(pd.DataFrame(status), "build status")

### 5 · Fi-ImageNet-1k (optional, when you have the authors' lists)

Its images are ImageNet validation images, already encoded above, so nothing new is encoded. Put
`fi_imagenet_1k_ood.txt` (655 images) and `fi_imagenet_1k_ambiguous.txt` (980) in any attached dataset.

In [ ]:
from oodlab.data.fi_imagenet import find_lists, read_val_ids, fi_imagenet_sets
import shutil
ood_list, amb_list = find_lists(ctx.input_roots)
if ood_list and cache.has("imagenet_val_all"):
    id_clean, fi_ood = fi_imagenet_sets(cache.load_set("imagenet_val_all"), read_val_ids(ood_list),
                                        read_val_ids(amb_list) if amb_list else [], logger=log)
    os.makedirs(os.path.join(cache.dir, "fi_imagenet_1k"), exist_ok=True)
    for p in [ood_list, amb_list]:
        if p:
            shutil.copy2(p, os.path.join(cache.dir, "fi_imagenet_1k", os.path.basename(p)))
else:
    print("Fi-ImageNet-1k lists not attached - skipped (expected until the data is released).")

### 6 · Acceptance check: live run vs cache replay

The **official** TANL postprocessor runs on images decoded and encoded on the fly; **our** TANL runs on
the cached features; both see the same stream (all NINCO images + 5,000 ImageNet images). The plan's
criterion: FPR95 and AUROC agree to two decimals. On GPU the scores should be bit-identical.

In [ ]:
live = None
if LIVE_CHECK and cache.has("ninco") and cache.has("imagenet_val_all"):
    try:   # a failure here must not lose the hours of encoding above
        from oodlab.official.live import live_vs_cache
        from oodlab.data.manifest import Manifest
        ninco_man = build_manifest("ninco", ctx.input_roots, ctx.dirs.scratch, imglist_dir, logger=log,
                                   strict_counts=not ctx.smoke)   # re-extracts NINCO if it was imported
        live = live_vs_cache(model, bank, Manifest.load(cache.manifests_dir, "imagenet_val_all"),
                             cache.load_set("imagenet_val_all"), ninco_man, cache.load_set("ninco"),
                             n_id=LIVE_CHECK_N_ID, device=ctx.device, num_workers=NUM_WORKERS, logger=log)
        oodlab.utils.write_json(live, ctx.path("live_vs_cache.json"))
        print("LIVE == CACHE (two decimals):", live["pass"], "| identical scores:", live["max_abs_score_diff"] == 0)
    except Exception as e:
        log.exception("live-vs-cache check failed (the cache itself is fine; re-run this cell or notebook 01 later)")
        live = {"error": f"{type(e).__name__}: {e}"}
    finally:
        cleanup(os.path.join(ctx.dirs.scratch, "extracted", "ninco"))
else:
    print("live check skipped")

### 7 · Summary

In [ ]:
summary = pd.DataFrame(cache.summary())
show(summary, "feature cache")
summary.to_csv(ctx.path("cache_summary.csv"), index=False)
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(cache.dir) for f in fs)
missing = [n for n in SETS if not cache.has(n)]
finish(ctx, {"cache_dir": cache.dir, "cache_GB": round(size / 1e9, 3), "missing_sets": missing,
             "live_check": live})
print(f"cache size {size / 1e9:.2f} GB; missing sets: {missing or 'none'}")
print("NEXT: 'Save Version' finished -> attach this notebook's output to notebooks 02-04.")